# Notebook 11 — Cold-Start Evaluation

**PharmaLens**

---

## Objective

Evaluate all models under **cold-start** settings:

- **Cold-Drug:** Test drugs are completely unseen during training
- **Cold-Target:** Test protein targets are completely unseen during training

This measures whether models can generalize to truly novel drugs or proteins — the most practically relevant evaluation for drug discovery.

In [ ]:
import os, sys, warnings, pickle
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import *
from src.evaluation.metrics import compute_regression_metrics
from src.evaluation.experiment_tracker import ExperimentTracker
from src.models.xgboost_model import XGBoostDTI
from src.models.baselines import get_baseline_models

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()
tracker = ExperimentTracker()
print("Setup complete ✓")

In [ ]:
# Load features
with open(PROCESSED_DATA_DIR / 'drug_features.pkl', 'rb') as f:
    drug_features = pickle.load(f)
with open(PROCESSED_DATA_DIR / 'protein_features.pkl', 'rb') as f:
    protein_features = pickle.load(f)

def build_feature_matrix(df):
    X_list, y_list = [], []
    for _, row in df.iterrows():
        if row['drug_id'] in drug_features and row['target_id'] in protein_features:
            combined = np.concatenate([drug_features[row['drug_id']], protein_features[row['target_id']]])
            X_list.append(combined)
            y_list.append(row['score'])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

## 1. Cold-Drug Evaluation

In [ ]:
# Cold-drug split
cd_train = pd.read_csv(SPLITS_DIR / 'cold_drug_train.csv')
cd_val = pd.read_csv(SPLITS_DIR / 'cold_drug_val.csv')
cd_test = pd.read_csv(SPLITS_DIR / 'cold_drug_test.csv')

X_cd_train, y_cd_train = build_feature_matrix(cd_train)
X_cd_val, y_cd_val = build_feature_matrix(cd_val)
X_cd_test, y_cd_test = build_feature_matrix(cd_test)

# Verify cold-start
train_drugs = set(cd_train['drug_id'].unique())
test_drugs = set(cd_test['drug_id'].unique())
overlap = train_drugs & test_drugs
print(f"Cold-Drug Split:")
print(f"  Train drugs: {len(train_drugs)}, Test drugs: {len(test_drugs)}")
print(f"  Drug overlap: {len(overlap)} (should be 0) ✓" if len(overlap) == 0 else f"  WARNING: {len(overlap)} drugs leaked!")

In [ ]:
# Train and evaluate models on cold-drug split
cold_drug_results = {}

# XGBoost
xgb = XGBoostDTI(n_estimators=1000, max_depth=7, learning_rate=0.1, early_stopping_rounds=50)
xgb.fit(X_cd_train, y_cd_train, X_cd_val, y_cd_val, verbose=False)
y_pred = xgb.predict(X_cd_test)
cold_drug_results['XGBoost'] = compute_regression_metrics(y_cd_test, y_pred)
tracker.log_experiment(model='XGBoost', dataset='KIBA', split='cold_drug', metrics=cold_drug_results['XGBoost'], seed=42)

# Random Forest
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=500, max_depth=None, n_jobs=-1, random_state=42)
rf.fit(X_cd_train, y_cd_train)
y_pred = rf.predict(X_cd_test)
cold_drug_results['RandomForest'] = compute_regression_metrics(y_cd_test, y_pred)
tracker.log_experiment(model='RandomForest', dataset='KIBA', split='cold_drug', metrics=cold_drug_results['RandomForest'], seed=42)

print("\nCold-Drug Results:")
for name, m in cold_drug_results.items():
    print(f"  {name:20s} — RMSE: {m['rmse']:.4f}  Pearson: {m['pearson']:.4f}  CI: {m['ci']:.4f}")

## 2. Cold-Target Evaluation

In [ ]:
ct_train = pd.read_csv(SPLITS_DIR / 'cold_target_train.csv')
ct_val = pd.read_csv(SPLITS_DIR / 'cold_target_val.csv')
ct_test = pd.read_csv(SPLITS_DIR / 'cold_target_test.csv')

X_ct_train, y_ct_train = build_feature_matrix(ct_train)
X_ct_val, y_ct_val = build_feature_matrix(ct_val)
X_ct_test, y_ct_test = build_feature_matrix(ct_test)

cold_target_results = {}

# XGBoost
xgb_ct = XGBoostDTI(n_estimators=1000, max_depth=7, learning_rate=0.1, early_stopping_rounds=50)
xgb_ct.fit(X_ct_train, y_ct_train, X_ct_val, y_ct_val, verbose=False)
y_pred = xgb_ct.predict(X_ct_test)
cold_target_results['XGBoost'] = compute_regression_metrics(y_ct_test, y_pred)
tracker.log_experiment(model='XGBoost', dataset='KIBA', split='cold_target', metrics=cold_target_results['XGBoost'], seed=42)

# Random Forest
rf_ct = RandomForestRegressor(n_estimators=500, max_depth=None, n_jobs=-1, random_state=42)
rf_ct.fit(X_ct_train, y_ct_train)
y_pred = rf_ct.predict(X_ct_test)
cold_target_results['RandomForest'] = compute_regression_metrics(y_ct_test, y_pred)
tracker.log_experiment(model='RandomForest', dataset='KIBA', split='cold_target', metrics=cold_target_results['RandomForest'], seed=42)

print("\nCold-Target Results:")
for name, m in cold_target_results.items():
    print(f"  {name:20s} — RMSE: {m['rmse']:.4f}  Pearson: {m['pearson']:.4f}  CI: {m['ci']:.4f}")

## 3. Comparison: Random vs Cold-Start

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

splits = ['Random', 'Cold-Drug', 'Cold-Target']

for idx, (metric, ylabel) in enumerate([('rmse', 'RMSE ↓'), ('pearson', 'Pearson r ↑')]):
    ax = axes[idx]
    x = np.arange(len(splits))
    w = 0.3
    
    # Get values from tracker for each model
    for i, model_name in enumerate(['XGBoost', 'RandomForest']):
        vals = []
        color = MODEL_COLORS.get(model_name, COLORS['primary'])
        # Random split value would come from tracker
        if model_name in cold_drug_results:
            vals.append(cold_drug_results.get(model_name, {}).get(metric, 0))  # placeholder for random
            vals.append(cold_drug_results[model_name][metric])
            vals.append(cold_target_results.get(model_name, {}).get(metric, 0))
            ax.bar(x + i*w - w/2, vals, w, label=model_name, color=color, alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels(splits)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} by Split Strategy', fontweight='bold')
    ax.legend()

plt.suptitle('Cold-Start Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cold_start_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Observations

1. **Performance drops in cold-start settings** — this is expected and normal. Models struggle more when they must predict for completely novel drugs or targets.

2. **Cold-drug vs Cold-target:** If cold-drug performance is better, the model generalizes drug features well. If cold-target is better, the model generalizes protein features well.

3. **Practical implication:** Cold-start performance is what matters for real drug discovery, where you want to screen NEW candidate molecules against known or novel targets.

**Next:** [Notebook 12 — XAI Analysis](./12_xai_analysis.ipynb)